# Chotu EDA · What exactly are we training on?

**EDA** (exploratory data analysis) means looking hard at the data before training on it. A model can only learn what's in its data. Every surprise you catch here (junk text, documents longer than the model can see, a lopsided vocabulary) is a surprise you won't have to debug in a loss curve later.

Everything below uses **live data**:
- our uploaded dataset `magnateadi/chotu-fineweb-edu-10bt-tokens` (the exact token shards the model will train on), and
- the original FineWeb-Edu `sample-10BT` documents they came from (for metadata like quality score and source website).

**How to use it:** read the explainer, run the cell, look at the picture, then answer the *Think about it* questions before moving on. Write answers in `docs/LEARNING.md`, or ask Claude: *"Teach me Socratically, one question at a time, diagrams over maths. Topic: …"*

| Part | Question it answers |
|---|---|
| A | How much data do we have, and is it evenly split? |
| B | What kind of documents are these? (quality, sources, years, length) |
| C | How does text become tokens, and how long are documents in tokens? |
| D | Which tokens dominate the vocabulary? |
| E | Is there junk: duplicates, boilerplate, code, other languages? |
| F | Is the validation set a fair sample of the training set? |
| G | What does all this mean for training? |

In [ ]:
# 0. Setup: download what we need from Hugging Face (about 1.6 GB with the defaults)
%pip install -q pandas matplotlib==3.11.2
%matplotlib inline
import os, sys, glob, json, re, hashlib, math, collections
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

REPO = next(p for p in [Path.cwd(), Path.cwd().parent, Path.home() / 'Chotu', Path.home() / 'apps' / 'Chotu'] if (p / 'chotu' / 'config.py').exists())
sys.path.insert(0, str(REPO))
from dotenv import load_dotenv; load_dotenv(REPO / '.env')
from huggingface_hub import hf_hub_download, HfFileSystem
from transformers import AutoTokenizer
from chotu.config import TOKENIZER_ID
from chotu import xray as X

DATASET = 'magnateadi/chotu-fineweb-edu-10bt-tokens'   # our private tokenized shards
RAW = 'HuggingFaceFW/fineweb-edu'                       # the public source documents
TRAIN_SHARDS = 1          # train shards to analyse (each ~1.5 GB, ~720M tokens). 1 is plenty for EDA.
ROW_GROUPS_PER_FILE = 3   # raw documents sampled: 14 files x 3 x 1,000 = 42,000 documents
CACHE = Path.home() / 'chotu-data' / 'eda'; CACHE.mkdir(parents=True, exist_ok=True)

def get(path):
    return Path(hf_hub_download(DATASET, path, repo_type='dataset', local_dir=CACHE))

report = get('reports/data_report.md')
done = [json.loads(get(f'shards/{i:03d}.done').read_text()) for i in range(14)]
val_paths = [get(f'shards/val_{i:03d}.bin') for i in range(14)]
train_paths = [get(f'shards/train_{i:03d}.bin') for i in range(TRAIN_SHARDS)]
tok = AutoTokenizer.from_pretrained(TOKENIZER_ID); tok.model_max_length = 10**9
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
print('repo:', REPO, '| cache:', CACHE)
print(f'downloaded: {len(val_paths)} val shards, {len(train_paths)} train shard(s), 14 .done files, report')

## A · The big picture: how much data, and how is it split?

The data script turned each of FineWeb-Edu's **14 source files** into one train shard and one validation shard, and wrote a small `.done` file with its counts. Reading all 14 gives exact totals without opening 20 GB.

**Look for:** are the files roughly equal? Is the validation share close to 0.5% in every file (it's chosen by a hash of each document's id, so it should be)?

In [ ]:
df_files = pd.DataFrame(done)
df_files['file'] = df_files['file'].str.extract(r'(\d{3})_')[0]
df_files['val_share'] = df_files.val_tokens / (df_files.train_tokens + df_files.val_tokens)
tot = df_files[['docs', 'train_tokens', 'val_tokens', 'val_docs']].sum()
display(Markdown(f'''| | total |
|---|---|
| documents | **{tot.docs:,}** |
| train tokens | **{tot.train_tokens:,}** ({tot.train_tokens/1e9:.2f}B) |
| validation tokens | **{tot.val_tokens:,}** ({tot.val_tokens/(tot.train_tokens+tot.val_tokens):.2%}) |
| validation documents | {tot.val_docs:,} |
| average tokens per document | {(tot.train_tokens+tot.val_tokens)/tot.docs:,.0f} |'''))
fig, ax = plt.subplots(1, 2, figsize=(15, 3.6))
ax[0].bar(df_files.file, df_files.train_tokens / 1e6, color='#60a5fa')
ax[0].set(title='Train tokens per source file (millions)', xlabel='source file')
ax[1].bar(df_files.file, df_files.val_share * 100, color='#f59e0b'); ax[1].axhline(0.5, color='k', ls='--', lw=1)
ax[1].set(title='Validation share per file (%), target 0.5', xlabel='source file', ylim=(0, 0.8))
plt.tight_layout(); plt.show()

**Think about it**
1. File 013 is much smaller than the others. Does that matter for training? (Hint: the training script reads all shards as one long stream.)
2. Why choose validation documents by hashing their id rather than taking "the last 0.5%"? What could go wrong with the last 0.5%?
3. 50M validation tokens is a lot of text. Why not use 1M? Why not 1B?

## B · What kind of documents are these?

FineWeb-Edu is built from **Common Crawl** (a public snapshot of the web, taken every few months; each snapshot is a *dump* like `CC-MAIN-2013-20`). Hugging Face cleaned it, then ran a **classifier that scores how educational each page is, from 0 to 5**. Only pages whose score **rounds to 3 or more** (a raw score of 2.5+, the red line below) were kept. So this is the "textbook-like" slice of the web.

Our token shards don't keep this metadata, so we sample the **original documents** (42,000 by default, spread evenly over all 14 files) and look at their quality score, language confidence, source website, crawl year and length.

In [ ]:
fs = HfFileSystem()
rng = np.random.default_rng(0)
cols = ['id', 'text', 'dump', 'url', 'language_score', 'token_count', 'score', 'int_score']
parts = []
for i in range(14):
    path = f'datasets/{RAW}/sample/10BT/{i:03d}_00000.parquet'
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(fs.open(path, 'rb'))
    for rg in sorted(rng.choice(pf.num_row_groups, ROW_GROUPS_PER_FILE, replace=False)):
        parts.append(pf.read_row_group(int(rg), columns=cols).to_pandas())
raw = pd.concat(parts, ignore_index=True)
raw['year'] = raw.dump.str.extract(r'CC-MAIN-(\d{4})')[0].astype(int)
raw['domain'] = raw.url.str.extract(r'https?://(?:www\.)?([^/]+)')[0]
raw['chars'] = raw.text.str.len()
print(f'{len(raw):,} sampled documents from {raw.dump.nunique()} crawl dumps, {raw.domain.nunique():,} websites')
raw[['dump', 'domain', 'score', 'int_score', 'language_score', 'token_count', 'chars']].head()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 8))
ax[0,0].hist(raw.score, bins=40, color='#10b981'); ax[0,0].axvline(2.5, color='r', ls='--', lw=1)
ax[0,0].set(title='Educational score (classifier, 0-5)', xlabel='score')
ax[0,1].bar(raw.int_score.value_counts().sort_index().index, raw.int_score.value_counts().sort_index().values, color='#10b981')
ax[0,1].set(title='Rounded score (only 3+ kept)', xlabel='int_score')
ax[0,2].hist(raw.language_score, bins=40, color='#a78bfa'); ax[0,2].set(title='Confidence the text is English', xlabel='language_score')
yr = raw.year.value_counts().sort_index(); ax[1,0].bar(yr.index, yr.values, color='#f59e0b'); ax[1,0].set(title='Crawl year (when the page was captured)')
ax[1,1].hist(np.log10(raw.chars.clip(lower=1)), bins=50, color='#60a5fa')
ax[1,1].set(title='Document length (log10 characters)', xlabel='10^x characters')
top = raw.domain.value_counts().head(15)[::-1]; ax[1,2].barh(top.index, top.values, color='#9ca3af'); ax[1,2].set(title='Top 15 websites in the sample')
plt.tight_layout(); plt.show()
print(f'median length: {raw.chars.median():,.0f} characters | 90% of documents under {raw.chars.quantile(.9):,.0f} | longest: {raw.chars.max():,}')
print(f'top 15 websites = {top.sum()/len(raw):.1%} of documents: the data is spread over a huge number of sites')

In [ ]:
def show(row, n=700):
    t = row.text[:n].replace('<', '&lt;')
    display(HTML(f"<div style='border:1px solid #ddd;border-radius:6px;padding:8px;margin:6px 0;font-family:sans-serif'>"
                 f"<b>score {row.score:.2f}</b> · {row.domain} · {row.dump} · {row.chars:,} chars<br>"
                 f"<div style='white-space:pre-wrap;font-family:monospace;font-size:12px;color:#333'>{t}…</div></div>"))
print('Lowest-scoring kept document:'); show(raw.loc[raw.score.idxmin()])
print('Highest-scoring document:'); show(raw.loc[raw.score.idxmax()])
print('Shortest document:'); show(raw.loc[raw.chars.idxmin()])

**Think about it**
1. Read the lowest- and highest-scoring documents. Would you agree with the classifier? What does a score really measure, and what can't it measure (for example, whether a fact is true)?
2. The crawl years span over a decade. What does that mean for what the model "knows" about recent events?
3. No single website dominates. Why is that good? What would happen if one site were 30% of the data?

## C · From text to tokens: how long are documents, really?

The model reads **tokens**, not characters. Two numbers matter here:
- **Characters (and bytes) per token**: how much text one token carries. We'll need **bytes per token** later, because we judge the model in *bits per byte*, a score that's fair across tokenizers.
- **Document length in tokens versus the 2,048-token context window**: the model can look back at most 2,048 tokens. Documents longer than that are cut into pieces, and the model never sees their beginning and end together.

In [ ]:
sample = raw.sample(2000, random_state=0)
ids = tok(sample.text.tolist(), add_special_tokens=False)['input_ids']
n_tok = np.array([len(i) for i in ids]); n_chr = sample.chars.values
n_byte = np.array([len(t.encode('utf-8')) for t in sample.text])
BYTES_PER_TOKEN = n_byte.sum() / n_tok.sum()
print(f'characters per token: {n_chr.sum()/n_tok.sum():.2f} | bytes per token: {BYTES_PER_TOKEN:.2f}')
print(f'our tokenizer vs GPT-2 on the same docs: {n_tok.sum()/sample.token_count.sum():.3f}x as many tokens')
display(HTML(X.token_lens(tok, sample.text.iloc[0][:300])))

In [ ]:
def doc_lengths(paths):
    out = []
    for p in paths:
        a = np.memmap(p, dtype=np.uint16, mode='r')
        ends = np.flatnonzero(a == 0)
        out.append(np.diff(np.concatenate([[-1], ends])))   # tokens per document, incl. the end marker
    return np.concatenate(out)
L = doc_lengths(train_paths)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
ax[0].hist(np.log10(L), bins=60, color='#60a5fa'); ax[0].axvline(np.log10(2048), color='r', ls='--')
ax[0].text(np.log10(2048), ax[0].get_ylim()[1]*0.9, '  2,048 context', color='r')
ax[0].set(title='Document length in tokens (log10)', xlabel='10^x tokens')
s = np.sort(L); cum_tokens = np.cumsum(s) / s.sum()
ax[1].plot(s, cum_tokens); ax[1].set_xscale('log'); ax[1].axvline(2048, color='r', ls='--')
ax[1].set(title='Share of all tokens in documents up to length x', xlabel='document length (tokens)', ylabel='cumulative share of tokens')
plt.tight_layout(); plt.show()
long = L > 2048
print(f'{len(L):,} documents in {len(train_paths)} train shard(s) | median {np.median(L):,.0f} tokens | mean {L.mean():,.0f}')
print(f'{long.mean():.1%} of documents are longer than 2,048 tokens, but they hold {L[long].sum()/L.sum():.1%} of all tokens')

### How training actually reads the data

Training doesn't feed documents one at a time. It treats the shards as **one long stream of tokens** and cuts it into windows of 2,048. So a window often contains the end of one document, an end-of-document marker, and the start of the next. Below are the first windows of a real shard: each coloured block is one document, and red lines are the end-of-document markers.

In [ ]:
a = np.memmap(train_paths[0], dtype=np.uint16, mode='r')
fig, ax = plt.subplots(figsize=(15, 2.6))
for w in range(6):
    win = np.asarray(a[w*2048:(w+1)*2048]); ends = np.flatnonzero(win == 0)
    starts = np.concatenate([[0], ends + 1]); stops = np.concatenate([ends + 1, [2048]])
    for k, (s0, s1) in enumerate(zip(starts, stops)):
        if s1 > s0: ax.barh(w, s1 - s0, left=s0, color=plt.cm.tab20(k % 20), edgecolor='white')
    for e in ends: ax.plot([e, e], [w-0.4, w+0.4], color='red', lw=1.5)
ax.set(yticks=range(6), yticklabels=[f'window {w}' for w in range(6)], xlabel='position in the 2,048-token window', xlim=(0, 2048))
ax.invert_yaxis(); ax.set_title('Real training windows: coloured blocks = documents, red = end-of-document marker'); plt.show()

**Think about it**
1. Most *documents* fit in 2,048 tokens, but a big share of *tokens* sit in long documents. Why do both numbers matter?
2. A window can start in the middle of a document. What does the model have to guess at the very start of such a window? Is that harmful or useful?
3. The model sees text from two unrelated documents in one window, separated by the marker. What must it learn about that marker?
4. Bytes per token is about 4.5. If the model's loss improves but bytes per token were different for another tokenizer, why would comparing raw loss be unfair?

## D · The vocabulary: which tokens dominate?

The tokenizer has 49,152 possible tokens, but usage is extremely uneven: a few tokens ("the", ",", ".") appear constantly and most appear rarely. This **long tail** shapes what the model learns first (common patterns) and what it struggles with (rare words, names, facts).

In [ ]:
counts, total = X.token_frequency(train_paths, max_tokens=300_000_000)
fig = X.plot_token_frequency(tok, counts, total); plt.show()
vocab = np.arange(49_152); pieces = [tok.convert_ids_to_tokens(int(i)) for i in vocab]
def share(mask): return counts[:49_152][mask].sum() / total
is_digit = np.array([p.strip('Ġ').isdigit() for p in pieces])
is_space = np.array([p in ('Ġ', 'Ċ', 'ĊĊ', 'ĉ') or set(p) <= set('ĠĊĉ') for p in pieces])
word_start = np.array([p.startswith('Ġ') for p in pieces])
print(f'digit tokens: {share(is_digit):.2%} of all tokens | whitespace/newline tokens: {share(is_space):.2%}')
print(f'tokens that start a new word (begin with a space): {share(word_start):.1%}')
print(f'vocabulary ids never seen in {total/1e6:.0f}M tokens: {(counts[:49_152]==0).sum():,} of 49,152')
print(f'special tokens in pretraining data: end-of-text {counts[0]:,} | <|im_start|> {counts[1]:,} | <|im_end|> {counts[2]:,} (chat markers should be 0 here)')

**Think about it**
1. Some vocabulary ids never appear in hundreds of millions of tokens. What happens to their embeddings during training?
2. The chat markers `<|im_start|>` and `<|im_end|>` should never appear in pretraining data. When will the model first see them, and why does that matter?
3. Digits are split one per token. Look at the digit share: is that a lot or a little for teaching arithmetic?

## E · Junk check: duplicates, boilerplate, code and other languages

Web data has junk even after filtering: the same page saved twice, cookie banners and footers repeated across sites, code, other languages. We check the 42,000 sampled documents. **Exact duplicates** share identical text; **repeated lines** reveal boilerplate.

In [ ]:
h = raw.text.map(lambda t: hashlib.md5(t.encode()).hexdigest())
h200 = raw.text.map(lambda t: hashlib.md5(t[:200].encode()).hexdigest())
lines = collections.Counter(l.strip() for t in raw.text for l in t.split('\n') if 20 <= len(l.strip()) <= 200)
non_ascii = raw.text.map(lambda t: sum(ord(c) > 127 for c in t) / max(1, len(t)))
probes = {
    'exact duplicate documents': h.duplicated().mean(),
    'same first 200 characters': h200.duplicated().mean(),
    'contain a URL': raw.text.str.contains(r'https?://').mean(),
    'look like code (def / {...;} / #include)': raw.text.str.contains(r'\bdef \w+\(|\{[^}]*;[^}]*\}|#include').mean(),
    'contain digits': raw.text.str.contains(r'\d').mean(),
    'more than 5% non-ASCII characters': (non_ascii > 0.05).mean(),
    'language score below 0.8': (raw.language_score < 0.8).mean(),
}
display(pd.DataFrame({'share of sampled documents': {k: f'{v:.2%}' for k, v in probes.items()}}))
print('Most repeated lines across documents (boilerplate candidates):')
for l, c in lines.most_common(12): print(f'  {c:4d} x  {l[:110]}')

**Think about it**
1. If the same document appears 100 times, what does the model learn from it, and what's the risk?
2. Look at the repeated lines. Are they harmful, harmless or useful? Would you remove them?
3. We skipped benchmark decontamination (see the "Cut on purpose" list in the plan). After seeing this data, how likely do you think it is that HellaSwag-style text is in here?

## F · Is the validation set a fair sample?

We'll judge the model by its loss on the validation tokens it never trains on. That only works if validation looks like training. We compare token frequencies and document lengths between the two.

In [ ]:
vc, vt = X.token_frequency(val_paths, max_tokens=60_000_000)
p_tr = counts[:49_152] / total; p_va = vc[:49_152] / vt
m = 0.5 * (p_tr + p_va)
def kl(a, b): k = (a > 0) & (b > 0); return float((a[k] * np.log2(a[k] / b[k])).sum())
js = 0.5 * kl(p_tr, m) + 0.5 * kl(p_va, m)
Lv = doc_lengths(val_paths)
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
k = (p_tr > 0) & (p_va > 0)
ax[0].loglog(p_tr[k], p_va[k], ',', alpha=0.4); lim = [p_tr[k].min(), p_tr[k].max()]; ax[0].plot(lim, lim, 'r--', lw=1)
ax[0].set(title=f'Each dot = one token: train vs validation frequency (JS divergence {js:.4f} bits)', xlabel='train frequency', ylabel='validation frequency')
ax[1].hist(np.log10(L), bins=60, density=True, alpha=0.5, label='train'); ax[1].hist(np.log10(Lv), bins=60, density=True, alpha=0.5, label='validation')
ax[1].set(title='Document length distribution (log10 tokens)'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f'validation: {vt:,} tokens, {len(Lv):,} documents, median {np.median(Lv):.0f} tokens (train median {np.median(L):.0f})')

**Think about it**
1. The dots hug the red line, and the JS divergence (a distance between two distributions; 0 means identical) is tiny. What would a validation set that *didn't* match look like on this chart?
2. The dots spread out at the low-frequency end. Is that a real difference, or just too few examples of rare tokens to measure?

## G · What this means for training

Turning the measurements above into the numbers that drive Session 5.

In [ ]:
params, batch = 180_925_056, 524_288
T = int(tot.train_tokens)
steps = T // batch
rows = {
    'train tokens': f'{T:,}',
    'tokens per parameter': f'{T/params:.0f} (compute-optimal is ~20; we train longer, which helps a small model)',
    'optimizer steps (one pass, 524,288-token batches)': f'{steps:,}',
    'warmup / stable / decay steps': f'~700 / {int(steps*0.8)-700:,} / {steps-int(steps*0.8):,}',
    'times the model sees each token': '1 (one pass, no repeats)',
    'bytes per token (for bits-per-byte)': f'{BYTES_PER_TOKEN:.2f}',
    'documents longer than the 2,048 context': f'{long.mean():.1%} of documents, {L[long].sum()/L.sum():.1%} of tokens',
    'H100 time at 15-25% MFU': f'{T*1.337e9/(989e12*0.25)/3600:.0f}-{T*1.337e9/(989e12*0.15)/3600:.0f} hours',
}
display(pd.DataFrame({'value': rows}))

## Summary: what you should now know

- **Size and split:** about 10.06B training tokens from 9.67M documents, with a 0.5% validation split that matches training closely.
- **Content:** educational-leaning English web pages from many years and a huge number of sites, filtered by a quality classifier. They're not checked for truth, and not decontaminated against benchmarks.
- **Tokens:** about 4.5 bytes per token; most documents fit in 2,048 tokens, but long documents hold a large share of the text and get cut into windows.
- **Vocabulary:** a steep long tail. Common tokens dominate, many ids are rare, and chat markers are absent until chat tuning.
- **Training plan:** about 19,000 steps, one pass, ~56 tokens per parameter.

**Before Session 3:** write your answers in `docs/LEARNING.md`, and note anything surprising in `docs/PROGRESS.md` under *Open questions*.